# **Python Project**
# **Password Manager**

In [ ]:
import sqlite3
from cryptography.fernet import Fernet

DB_NAME = "passwords.db"

# Generate or load encryption key
def load_key():
    try:
        with open("secret.key", "rb") as key_file:
            return key_file.read()
    except FileNotFoundError:
        key = Fernet.generate_key()
        with open("secret.key", "wb") as key_file:
            key_file.write(key)
        return key

key = load_key()
cipher = Fernet(key)

def get_connection():
    return sqlite3.connect(DB_NAME, timeout=10)

def init_db():
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS vault (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            app TEXT NOT NULL,
            username TEXT NOT NULL,
            password TEXT NOT NULL
        )
        """)
        conn.commit()

def add_password():
    app = input("Enter app/website: ")
    username = input("Enter username: ")
    password = input("Enter password: ")
    encrypted_pw = cipher.encrypt(password.encode()).decode()
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("INSERT INTO vault (app, username, password) VALUES (?, ?, ?)",
                       (app, username, encrypted_pw))
        conn.commit()
        print("Password saved successfully!")

def view_passwords():
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM vault")
        rows = cursor.fetchall()
        if rows:
            for row in rows:
                decrypted_pw = cipher.decrypt(row[3].encode()).decode()
                print(f"ID: {row[0]} | App: {row[1]} | User: {row[2]} | Password: {decrypted_pw}")
        else:
            print("No passwords stored.")

def search_password():
    app = input("Enter app/website to search: ")
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM vault WHERE app=?", (app,))
        row = cursor.fetchone()
        if row:
            decrypted_pw = cipher.decrypt(row[3].encode()).decode()
            print(f"App: {row[1]} | User: {row[2]} | Password: {decrypted_pw}")
        else:
            print("No entry found.")

def update_password():
    acc_id = int(input("Enter ID to update: "))
    new_pw = input("Enter new password: ")
    encrypted_pw = cipher.encrypt(new_pw.encode()).decode()
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("UPDATE vault SET password=? WHERE id=?", (encrypted_pw, acc_id))
        conn.commit()
        print("Password updated successfully!")

def delete_password():
    acc_id = int(input("Enter ID to delete: "))
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM vault WHERE id=?", (acc_id,))
        conn.commit()
        print("Entry deleted successfully!")

def menu():
    while True:
        print("\n--- Password Manager ---")
        print("1. Add Password")
        print("2. View All Passwords")
        print("3. Search Password by App")
        print("4. Update Password")
        print("5. Delete Password")
        print("6. Exit")

        choice = input("Enter choice: ")
        if choice == "1":
            add_password()
        elif choice == "2":
            view_passwords()
        elif choice == "3":
            search_password()
        elif choice == "4":
            update_password()
        elif choice == "5":
            delete_password()
        elif choice == "6":
            print("Exiting...")
            break
        else:
            print("Invalid choice, try again.")

if __name__ == "__main__":
    init_db()
    menu()




--- Password Manager ---
1. Add Password
2. View All Passwords
3. Search Password by App
4. Update Password
5. Delete Password
6. Exit
Enter choice: 1
Enter app/website: whatsapp 
Enter username: kartik magar
Enter password: 12102007
Password saved successfully!

--- Password Manager ---
1. Add Password
2. View All Passwords
3. Search Password by App
4. Update Password
5. Delete Password
6. Exit
Enter choice: 1
Enter app/website: Instagram 
Enter username: pooja magar
Enter password: Pass@123
Password saved successfully!

--- Password Manager ---
1. Add Password
2. View All Passwords
3. Search Password by App
4. Update Password
5. Delete Password
6. Exit
Enter choice: 1
Enter app/website: Youtube
Enter username: premal 
Enter password: 147369
Password saved successfully!

--- Password Manager ---
1. Add Password
2. View All Passwords
3. Search Password by App
4. Update Password
5. Delete Password
6. Exit
Enter choice: 1
Enter app/website: Snapchart 
Enter username: Sanskruti 
Enter pa